In [0]:
%run ./03_pipeline_models

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator


def evaluate_population(population, df, use_pca=False, use_weight=False):

    print(f"\nEvaluating {len(population)} chromosomes...")

    results = []

    # ── PREPROCESS ONCE ──────────────────────────────────────
    # Split first, then fit preprocessing on train only
    train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=42)

    prep_stages  = build_pipeline(df, use_pca)
    prep_pipeline = Pipeline(stages=prep_stages)
    prep_fitted  = prep_pipeline.fit(train_raw)

    # Transform and CACHE — this is the key line
    # Without cache, Spark recomputes from raw data every time
    train_prep = prep_fitted.transform(train_raw).cache()
    test_prep  = prep_fitted.transform(test_raw).cache()

    # Force materialisation right now so cache is warm before the loop
    train_count = train_prep.count()
    test_count  = test_prep.count()
    print(f"Preprocessed and cached: {train_count} train / {test_count} test rows")

    # ── EVALUATE EACH CHROMOSOME (model only, no preprocessing) ──
    for model_type, params in population:

        model  = build_model(model_type, params, use_weight)

        try:
            fitted = model.fit(train_prep)
            preds  = fitted.transform(test_prep)

            auc       = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(preds)
            pr_auc    = BinaryClassificationEvaluator(metricName="areaUnderPR").evaluate(preds)
            accuracy  = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(preds)
            precision = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(preds)
            recall    = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(preds)
            f1        = MulticlassClassificationEvaluator(metricName="f1").evaluate(preds)

            print("\n==============================")
            print(f"Model     : {model_type}")
            print(f"Params    : {params}")
            print(f"AUC       : {auc:.4f}")
            print(f"PR-AUC    : {pr_auc:.4f}")
            print(f"Accuracy  : {accuracy:.4f}")
            print(f"Precision : {precision:.4f}")
            print(f"Recall    : {recall:.4f}")
            print(f"F1 Score  : {f1:.4f}")
            print("==============================\n")

            fitness = 0.6 * auc + 0.4 * f1
            results.append((fitness, (model_type, params)))

        except Exception as e:
            # If a model config crashes (e.g. bad param combo), skip it gracefully
            print(f"SKIPPED {model_type} {params} — Error: {e}")
            results.append((0.0, (model_type, params)))

    # ── RELEASE CACHE when done with this generation ──────────
    train_prep.unpersist()
    test_prep.unpersist()

    return results